<a href="https://colab.research.google.com/github/Darwisher/Big-Data-Group_Activity/blob/main/Lab_Activity_3_Your_Data%3F_Our_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("NBA Stats Analysis").getOrCreate()

df = spark.read.csv(
    "/content/2021-2022 NBA Player Stats - Regular.csv",
    header=True,
    inferSchema=True,
    sep=";"
)

df.show()
df.printSchema()


+---+--------------------+---+---+---+---+---+----+----+----+-----+---+---+-----+---+----+-----+-----+---+----+-----+---+---+----+---+---+---+---+---+----+
| Rk|              Player|Pos|Age| Tm|  G| GS|  MP|  FG| FGA|  FG%| 3P|3PA|  3P%| 2P| 2PA|  2P%| eFG%| FT| FTA|  FT%|ORB|DRB| TRB|AST|STL|BLK|TOV| PF| PTS|
+---+--------------------+---+---+---+---+---+----+----+----+-----+---+---+-----+---+----+-----+-----+---+----+-----+---+---+----+---+---+---+---+---+----+
|  1|    Precious Achiuwa|  C| 22|TOR| 73| 28|23.6| 3.6| 8.3|0.439|0.8|2.1|0.359|2.9| 6.1|0.468|0.486|1.1| 1.8|0.595|2.0|4.5| 6.5|1.1|0.5|0.6|1.2|2.1| 9.1|
|  2|        Steven Adams|  C| 28|MEM| 76| 75|26.3| 2.8| 5.1|0.547|0.0|0.0|  0.0|2.8| 5.0|0.548|0.547|1.4| 2.6|0.543|4.6|5.4|10.0|3.4|0.9|0.8|1.5|2.0| 6.9|
|  3|         Bam Adebayo|  C| 24|MIA| 56| 56|32.6| 7.3|13.0|0.557|0.0|0.1|  0.0|7.3|12.9|0.562|0.557|4.6| 6.1|0.753|2.4|7.6|10.1|3.4|1.4|0.8|2.6|3.1|19.1|
|  4|        Santi Aldama| PF| 21|MEM| 32|  0|11.3| 1.7| 4.1|0.4

In [9]:
#Partition Strategy #1 – Partition by Team
#This groups data by team to improve performance when analyzing team statistics.

df_team_partition = df.repartition("Tm")

df_team_partition.write.partitionBy("Tm").csv("output/team_partition")

In [10]:
#Transformation Operations
#Filter players from one team
df.filter(df.Tm == "LAL").show()

+---+-------------------+---+---+---+---+---+----+----+----+-----+---+---+-----+---+----+-----+-----+---+---+-----+---+---+---+---+---+---+---+---+----+
| Rk|             Player|Pos|Age| Tm|  G| GS|  MP|  FG| FGA|  FG%| 3P|3PA|  3P%| 2P| 2PA|  2P%| eFG%| FT|FTA|  FT%|ORB|DRB|TRB|AST|STL|BLK|TOV| PF| PTS|
+---+-------------------+---+---+---+---+---+----+----+----+-----+---+---+-----+---+----+-----+-----+---+---+-----+---+---+---+---+---+---+---+---+----+
| 14|    Carmelo Anthony| PF| 37|LAL| 69|  3|26.0| 4.6|10.5|0.441|2.2|5.8|0.375|2.5| 4.7|0.521|0.544|1.9|2.3| 0.83|0.9|3.3|4.2|1.0|0.7|0.8|0.9|2.4|13.3|
| 18|       Trevor Ariza| SF| 36|LAL| 24| 11|19.3| 1.4| 4.1|0.333|0.8|3.1| 0.27|0.5| 1.0| 0.52|0.434|0.4|0.8|0.556|0.4|3.0|3.4|1.1|0.5|0.3|0.5|0.8| 4.0|
| 19|      D.J. Augustin| PG| 34|LAL| 21|  0|17.8| 1.9| 4.1|0.453|1.4|3.2|0.426|0.5| 0.9|0.556|0.622|0.2|0.2|  1.0|0.2|1.1|1.3|1.6|0.3|0.0|0.5|1.0| 5.3|
| 39|      Kent Bazemore| SF| 32|LAL| 39| 14|14.0| 1.2| 3.6|0.324|0.7|2.1|0.363|0.

In [11]:
#Average points per team
from pyspark.sql.functions import col, sum, max, round

df.filter(col("Tm") != "TOT") \
  .groupBy("Tm") \
  .agg(
      round(sum(col("PTS") * col("G")) / max("G"), 2).alias("TeamAvgPoints")
  ).show()

+---+-------------+
| Tm|TeamAvgPoints|
+---+-------------+
|GSW|        111.1|
|LAL|       118.03|
|DET|       104.88|
|NYK|       107.86|
|CHO|        118.3|
|LAC|       109.72|
|UTA|       117.98|
|BOS|       119.08|
|TOR|       112.09|
|SAS|       123.76|
|POR|       136.12|
|DEN|       123.26|
|BRK|       114.29|
|DAL|       108.04|
|CLE|       119.52|
|MIA|       114.23|
|OKC|       123.35|
|PHO|       114.84|
|MIN|       120.43|
|MEM|       121.48|
+---+-------------+
only showing top 20 rows
